# セミナーII 第2回 実践：写真の中の物を切り抜く（SAM 2）

写真の中の「これ」と指定した物を、輪郭に沿って切り抜くモデル **SAM 2**（Segment Anything Model 2）を動かします。点や枠で指定するだけで、何の物かを知らなくても切り抜けます。中では、講義で見た Transformer のアテンションが使われています。

難しいことは考えずに、上のセルから順に実行してください。コードを読めなくてもかまいません。何ができて、何が苦手かを、自分の目で確かめることを目標にします。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します。セルの左の ▶ を押すか、セルを選んで **Shift + Enter** を押すと実行できます。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約45分（うち計算 約5分）です。各節の見出しの下に目安を書いています。

**設定の選び方**：セルの右に入力欄やスライダーが出るセルでは、そこで値を選んでから実行します（コードの `= ` の後ろを書き換えても同じです）。**関数を定義するセル**は、コードを畳んであります。実行するだけでかまいません（ダブルクリックで中身を見られます）。

**位置の指定**：点や枠の位置は、画像の左上を (0, 0)、右下を (1, 1) とした数で指定します。図の目盛りを見て決めてください。

**うまくいかないとき**：しばらく操作しないと接続が切れ、読み込んだモデルや画像が消えます。`NameError` が出たときなどは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | ライブラリとモデルを読み込む |
| 2. 手法を知る | セグメンテーションとは何か。SAM の仕組みと、学習のしかた |
| 3. 例の画像で試す | 点を1つ置いて切り抜く |
| 4. 自分の画像で試す | 自分で撮った写真を入れる |
| 5. やってみよう | 点を足す・除く、枠で指定する、言葉で指定する、全部を自動で切り抜く |
| 6. 活用 | 背景を差し替える、物だけ色を残す、数えて測る、動画で追いかける |
| 7. まとめ | 分かったことと、苦手なこと |

## 1. 準備

目安：約3分（うち計算 約1分）

使うライブラリを読み込み、計算に使う装置（`device`）を決めます。`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import importlib.util
import io
import subprocess
import sys
import time

# Colab に入っていないライブラリがあれば入れる
for package in ["transformers", "pooch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as ShowImage, display
from PIL import Image
from skimage import data
from transformers import Sam2Model, Sam2Processor, Sam2VideoModel, Sam2VideoProcessor, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32  # GPU では半分の精度で速く計算する
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

このノートブックで使う関数を定義します。モデルの読み込み、切り抜き、図の表示などです。

In [ ]:
# @title 関数を定義する（実行するだけでよい。ダブルクリックでコードを表示） { display-mode: "form" }
SAM_ID = "facebook/sam2.1-hiera-small"             # SAM 2.1 の Small
DETECTOR_ID = "google/owlv2-base-patch16-ensemble"  # 言葉で物を探す OWLv2（5-3 で使う）
COLORS = np.array([[230, 60, 60], [60, 110, 235], [40, 175, 80], [235, 165, 30], [165, 70, 210],
                   [30, 190, 200], [240, 110, 180], [130, 130, 40]])
_embeddings = {}
_detector = None


def load_detector():
    global _detector
    if _detector is None:
        _detector = pipeline("zero-shot-object-detection", model=DETECTOR_ID, device=device, dtype=dtype)
    return _detector


def to_pixels(image, xy):
    # (0〜1, 0〜1) の位置を、画素の座標にする
    return [float(xy[0]) * (image.width - 1), float(xy[1]) * (image.height - 1)]


def embed(image):
    # 画像エンコーダで、画像の特徴を計算する（重い処理なので、画像ごとに1回だけ）
    key = id(image)
    if key not in _embeddings:
        pixel_values = sam_processor(images=image, return_tensors="pt")["pixel_values"].to(device, dtype)
        with torch.no_grad():
            _embeddings[key] = sam.get_image_embeddings(pixel_values)
    return _embeddings[key]


def segment(image, points=None, box=None, multimask=True):
    # 点（[(x, y, ラベル), ...]。ラベルは 1：含める、0：除く）か枠（(x0, y0, x1, y1)）で指定した物を切り抜く
    # 位置は 0〜1。マスク（候補の数×縦×横）と、それぞれの品質の予測（IoU）を返す
    kwargs = {}
    if points:
        kwargs["input_points"] = [[[to_pixels(image, p[:2]) for p in points]]]
        kwargs["input_labels"] = [[[int(p[2]) for p in points]]]
    if box is not None:
        kwargs["input_boxes"] = [[to_pixels(image, box[:2]) + to_pixels(image, box[2:])]]
    inputs = sam_processor(images=image, return_tensors="pt", **kwargs).to(device)
    with torch.no_grad():
        outputs = sam(image_embeddings=embed(image), multimask_output=multimask,
                      **{k: v for k, v in inputs.items() if k.startswith("input_")})
    masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0][0]
    scores = outputs.iou_scores[0, 0].float().cpu()
    order = scores.argsort(descending=True)  # 品質の予測が高い順に並べる
    return masks[order].numpy(), scores[order].numpy()


def segment_grid(image, n=16, min_score=0.8, batch=64):
    # 画像に n×n の点を並べて1つずつ切り抜き、重なりを除いて、全部の物のマスクを返す
    xs = (np.arange(n) + 0.5) / n
    grid = [(x, y) for y in xs for x in xs]
    found = []
    for i in range(0, len(grid), batch):
        chunk = [to_pixels(image, p) for p in grid[i:i + batch]]
        inputs = sam_processor(images=image, input_points=[[[p] for p in chunk]],
                               input_labels=[[[1] for _ in chunk]], return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = sam(image_embeddings=embed(image), input_points=inputs["input_points"],
                          input_labels=inputs["input_labels"], multimask_output=True)
        masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
        scores = outputs.iou_scores[0].float().cpu()
        for j, k in enumerate(scores.argmax(dim=1)):  # 点ごとに、品質の予測が一番高い候補
            found.append((scores[j, k].item(), masks[j, k].numpy()))
    found.sort(key=lambda f: -f[0])
    kept = []
    for score, mask in found:
        area = mask.mean()
        if score < min_score or area < 0.0005 or area > 0.5:  # 品質が低いもの、小さすぎ・大きすぎるもの
            continue
        if any((mask & other).sum() / (mask | other).sum() > 0.7 for other in kept):  # すでにある物と同じ
            continue
        kept.append(mask)
    return kept


def overlay(image, masks, alpha=0.55):
    # マスクを色で重ね、輪郭を描いた画像を返す
    out = np.array(image.convert("RGB")).astype(np.float32)
    for i, mask in enumerate(masks):
        color = COLORS[i % len(COLORS)]
        out[mask] = out[mask] * (1 - alpha) + color * alpha
    out = out.astype(np.uint8)
    for i, mask in enumerate(masks):
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, contours, -1, [int(c) for c in COLORS[i % len(COLORS)]], 2)
    return out


def draw_prompts(ax, image, points=None, box=None):
    # 指定した点（緑の星：含める、赤の×：除く）と枠を描く
    for x, y, label in points or []:
        px, py = to_pixels(image, (x, y))
        ax.plot(px, py, "*" if label else "X", color="lime" if label else "red", markersize=18,
                markeredgecolor="black", markeredgewidth=1.2)
    if box is not None:
        (x0, y0), (x1, y1) = to_pixels(image, box[:2]), to_pixels(image, box[2:])
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow", lw=2.5))


def show_candidates(image, masks, scores, points=None, box=None, title=""):
    # 候補のマスクを、品質の予測（IoU）と並べて表示する
    fig, axes = plt.subplots(1, len(masks), figsize=(5.2 * len(masks), 4.6), squeeze=False)
    for i, (ax, mask, score) in enumerate(zip(axes[0], masks, scores)):
        ax.imshow(overlay(image, [mask]))
        draw_prompts(ax, image, points, box)
        ax.set_title(f"{title}  candidate {i + 1}: predicted IoU {score:.2f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_grid(image, title=""):
    # 位置を決めやすいように、0〜1 の目盛りを付けて表示する
    fig, ax = plt.subplots(figsize=(7, 7 * image.height / image.width))
    ax.imshow(image, extent=(0, 1, 1, 0))
    ax.set_xticks(np.linspace(0, 1, 11))
    ax.set_yticks(np.linspace(0, 1, 11))
    ax.grid(color="white", alpha=0.5, lw=0.6)
    ax.set_aspect(image.height / image.width)
    ax.set_title(title)
    plt.show()


def fit_size(image, max_side=1024):
    # 大きすぎる写真は縮小する（計算と表示を速くするため）
    image = image.convert("RGB")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


def show_gif(frames, duration=100):
    # 画像の列を、くり返し再生する GIF にして表示する
    buffer = io.BytesIO()
    frames = [Image.fromarray(f) for f in frames]
    frames[0].save(buffer, format="GIF", save_all=True, append_images=frames[1:], duration=duration, loop=0)
    display(ShowImage(data=buffer.getvalue(), format="gif"))


print("関数を定義しました")

SAM 2.1 の Small（約3900万パラメータ）を読み込みます。初回は重み（約200MB）をダウンロードします。`HF_TOKEN` に関する警告や、`sam2_video` の型についての警告が表示されても、動作には問題ありません（動画用と共通の重みを、画像用として読むためです）。

- モデル：[facebook/sam2.1-hiera-small](https://huggingface.co/facebook/sam2.1-hiera-small)（Apache-2.0）
- 論文：N. Ravi et al., "SAM 2: Segment Anything in Images and Videos," ICLR 2025（[arXiv:2408.00714](https://arxiv.org/abs/2408.00714)）。もとになった SAM は A. Kirillov et al., "Segment Anything," ICCV 2023（[arXiv:2304.02643](https://arxiv.org/abs/2304.02643)）

In [ ]:
sam_processor = Sam2Processor.from_pretrained(SAM_ID)
sam = Sam2Model.from_pretrained(SAM_ID, dtype=dtype).to(device).eval()
print(f"パラメータの数: {sum(p.numel() for p in sam.parameters()) / 1e6:.1f}M")

## 2. 手法を知る

目安：約10分

### 2-1. セグメンテーションとは

画像の中の物を扱う問題には、答えの細かさによって次の種類があります。

| 問題 | 答え | 例 |
| --- | --- | --- |
| 分類 | 画像1枚に1つのクラス | 第1回：37品種のどれか |
| 物体検出 | 物ごとに枠（四角）とクラス | 5-3 の OWLv2 |
| セグメンテーション | **画素ごと**に、どの物（クラス）に属するか | このノートブック |

セグメンテーションは、奥行きの推定と同じく、画素ごとに答えを出す「密な予測」です。クラスごとに塗り分けるもの（semantic segmentation）や、同じクラスでも物ごとに分けるもの（instance segmentation）があります。

**SAM**（Segment Anything Model）は、決まったクラスを塗り分けるのではなく、**「どれ」を指定すると、その物の領域（マスク）を返す**モデルです。指定（プロンプト）には、点、枠、おおまかなマスクを使えます。何の物かは答えませんが、その代わり、学習で見たことのない種類の物でも切り抜けます。

### 2-2. SAM の仕組み

SAM は3つの部品からできています。

| 部品 | 処理 | 計算の重さ |
| --- | --- | --- |
| ① 画像エンコーダ | 画像全体を読み、画像の特徴（埋め込み）を計算する。SAM 2 では **Hiera** という ViT を使う | 重い。画像ごとに1回 |
| ② プロンプトエンコーダ | 点や枠の位置を、ベクトル（トークン）にする | 軽い |
| ③ マスクデコーダ | 指定のトークンと画像の特徴の間で、**両方向のアテンション**（指定 → 画像、画像 → 指定）を計算し、マスクと、その品質の予測（予測 IoU）を出す | 軽い。指定ごとに計算 |

**Hiera** は、層が進むにつれて、トークンの数を減らして（解像度を下げて）いく階層的な ViT です。第1回の CNN と同じように、粗い特徴と細かい特徴を両方取り出せます。ラベルなしで、画像の一部を隠して元に戻す学習（MAE）で事前学習しています。

重い ① は画像ごとに1回だけ計算し、② と ③ は指定のたびに計算します。SAM の論文では、② と ③ が Web ブラウザの CPU でも約50ミリ秒で動くとしています。点を置くとすぐに切り抜ける、対話的な使い方ができるのはこのためです。

**あいまいさ**：点を1つ置いただけでは、どこまでを指すのかが決まらないことがあります。バイクのタイヤに点を置いたら、タイヤだけか、車輪か、バイク全体か。SAM は、1つの指定に対して**3つの候補**を出し、それぞれの品質の予測を付けます。3. で確かめます。

下のセルで、① と ③ の計算時間を比べます。

In [ ]:
motorcycle = Image.fromarray(data.stereo_motorcycle()[0])  # 室内のバイク（Middlebury 2014 ステレオデータセット）

pixel_values = sam_processor(images=motorcycle, return_tensors="pt")["pixel_values"].to(device, dtype)
print("画像エンコーダへの入力:", tuple(pixel_values.shape[-2:]), "画素（縦横 1024 にそろえる）")
with torch.no_grad():
    sam.get_image_embeddings(pixel_values)  # 1回目は準備に時間がかかるので、測る前に1回動かす
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    features = sam.get_image_embeddings(pixel_values)  # ① 画像エンコーダ
    if device == "cuda":
        torch.cuda.synchronize()
    encode_ms = (time.time() - start) * 1000
for f in features:
    print("  画像の特徴:", tuple(f.shape[1:]), "（チャンネル, 縦, 横）")

point = sam_processor(images=motorcycle, input_points=[[[[300.0, 300.0]]]], input_labels=[[[1]]],
                      return_tensors="pt").to(device)
with torch.no_grad():
    sam(image_embeddings=features, input_points=point["input_points"], input_labels=point["input_labels"])  # 測る前に1回
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    sam(image_embeddings=features, input_points=point["input_points"], input_labels=point["input_labels"])  # ②③
    if device == "cuda":
        torch.cuda.synchronize()
    decode_ms = (time.time() - start) * 1000
print(f"① 画像エンコーダ: {encode_ms:.0f} ms（画像ごとに1回）")
print(f"②③ 点1つからマスクを出す: {decode_ms:.1f} ms（点を置くたび）")

画像の特徴は、細かい順に 256×256、128×128、64×64 の3段階です。階層的な ViT から、大きさの違う特徴を取り出していることが分かります。

### 2-3. どうやって学習したか

学習には、大量の画像と、物の輪郭をなぞったマスクが必要です。SAM の論文では、モデルとデータを一緒に育てる **データエンジン** で、データセット **SA-1B**（**1100万枚**の画像、**11億個**のマスク。1枚あたり平均約100個）を作りました。

| 段階 | 内容 |
| --- | --- |
| ① 人がモデルの助けを借りる | 注釈をつける人が、SAM を組み込んだツールで点をクリックしてマスクを作る。集まったデータで SAM を学習し直す |
| ② 半自動 | SAM が自信をもって切り抜けた物は自動で、残りを人が注釈する |
| ③ 全自動 | 画像に**格子状に点を並べて** SAM に切り抜かせ、良いマスクだけを残す（5-4 で同じことをする） |

**SAM 2** は、SAM を動画にも使えるように広げたモデルです。動画では、前のフレームの結果を**メモリ**にためておき、新しいフレームを読むときに、アテンションでメモリを参照して、同じ物を追いかけます（6-4 で試す）。学習には、動画のデータセット SA-V（5万本の動画、3550万個のマスク）も使っています。論文では、画像でも SAM より精度が高く、6倍速いとしています。

## 3. 例の画像で試す

目安：約3分

4枚の写真に、点（緑の星）を1つずつ置いて切り抜きます。SAM が出す3つの候補を、品質の予測（predicted IoU。1 に近いほど、モデルが良いと考えているマスク）の高い順に並べます。

| 名前 | 写真 | 出典 |
| --- | --- | --- |
| `motorcycle` | 室内のバイク | Middlebury 2014 ステレオデータセット（D. Scharstein et al., GCPR 2014） |
| `astronaut` | 宇宙飛行士の肖像 | NASA（パブリックドメイン） |
| `coffee` | テーブルの上のコーヒー | 撮影 Rachel Michetti（CC0） |
| `cat` | 猫の顔 | 撮影 Stefan van der Walt（CC0） |
| `coins` | 古代ギリシャの硬貨（6-3 で使う） | Brooklyn Museum（scikit-image の説明では、著作権の制限は知られていない） |

すべて scikit-image に入っている写真です。

In [ ]:
images = {
    "motorcycle": motorcycle,
    "astronaut": Image.fromarray(data.astronaut()),
    "coffee": Image.fromarray(data.coffee()),
    "cat": Image.fromarray(data.chelsea()),
    "coins": Image.fromarray(data.coins()).convert("RGB"),
}
EXAMPLE_POINTS = {  # 名前: (x, y)。位置は 0〜1
    "motorcycle": (0.25, 0.65),  # 排気管のあたり
    "astronaut": (0.45, 0.65),   # 宇宙服
    "coffee": (0.45, 0.45),      # カップの中のコーヒー
    "cat": (0.58, 0.82),         # 猫の鼻
}
for name, (x, y) in EXAMPLE_POINTS.items():
    points = [(x, y, 1)]
    masks, scores = segment(images[name], points=points)
    show_candidates(images[name], masks, scores, points=points, title=name)

**見どころ**：同じ1つの点でも、候補によって切り抜く範囲が違います。コーヒーでは、中のコーヒーだけの候補と、カップまで含む候補が出ます。猫では、鼻だけの候補と、猫全体の候補が出ます。バイクでは、排気管の部品の候補と、バイク全体の候補が出ます。

品質の予測が一番高い候補が、いつも自分の意図と同じとは限りません。バイク全体の候補は、品質の予測がかなり低く、輪郭も粗くなっています。このノートブックで使っているのは一番小さい Small のモデルなので、大きなモデルならもっとよくなることがあります。意図を伝えるには、点を足したり、枠で囲んだりします（5-1、5-2）。

## 4. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD` にチェックを入れて実行すると、アップロードのボタンが表示されます（Colab のみ）。アップロードした画像は、ファイル名（拡張子を除く）の名前で、5. と 6. でも使えます。目盛りを付けて表示するので、次のセルで点を置く位置を決めてください。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files

    for filename, content in files.upload().items():
        name = filename.rsplit(".", 1)[0]
        images[name] = fit_size(Image.open(io.BytesIO(content)))
        show_grid(images[name], name)
print("使える画像の名前:", list(images))

`IMAGE` に画像の名前を選び（自分の画像は名前を入力し）、`X` と `Y` で点の位置を決めて実行します。

In [ ]:
IMAGE = "motorcycle"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
X = 0.45  # @param {type:"slider", min:0, max:1, step:0.01}
Y = 0.35  # @param {type:"slider", min:0, max:1, step:0.01}

points = [(X, Y, 1)]
masks, scores = segment(images[IMAGE], points=points)
show_candidates(images[IMAGE], masks, scores, points=points, title=IMAGE)

## 5. やってみよう：指定のしかたを変える

目安：約12分（うち計算 約1分）

学習はしません。指定（プロンプト）のしかたを変えて、切り抜き方がどう変わるかを見ます。

### 5-1. 点を足す・除く

点は何個でも置けます。`(x, y, 1)` は「ここを含める」点（緑の星）、`(x, y, 0)` は「ここは除く」点（赤の×）です。例では、宇宙飛行士の宇宙服と顔に「含める」点を置いて人物全体を指定し、抱えたヘルメットに「除く」点を置いています。点を足したり、ラベルを変えたり（ヘルメットも含めるなど）してみましょう。

点が2つ以上のときは、指定のあいまいさが減るので、候補を1つだけ出します（`multimask=False`）。

In [ ]:
IMAGE = "astronaut"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
POINTS = [(0.45, 0.65, 1), (0.40, 0.25, 1), (0.62, 0.86, 0)]  # (x, y, ラベル)。1：含める、0：除く

masks, scores = segment(images[IMAGE], points=POINTS, multimask=len(POINTS) == 1)
show_candidates(images[IMAGE], masks, scores, points=POINTS, title=IMAGE)

### 5-2. 枠で指定する

物を囲む枠（黄色）で指定すると、点よりあいまいさが少なくなります。`BOX` は (左上の x, 左上の y, 右下の x, 右下の y) です。例では、3. の点1つではうまく取れなかったバイク全体を、枠で囲んで指定します。

In [ ]:
IMAGE = "motorcycle"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
BOX = (0.10, 0.15, 0.92, 0.92)  # (x0, y0, x1, y1)。位置は 0〜1

masks, scores = segment(images[IMAGE], box=BOX, multimask=False)
show_candidates(images[IMAGE], masks, scores, box=BOX, title=IMAGE)

### 5-3. 言葉で指定する（OWLv2 ＋ SAM 2）

SAM 2 は言葉では指定できません。そこで、言葉で物を探して枠を出すモデル **OWLv2** と組み合わせます。

1. **OWLv2** が、言葉（英語）で指定した物を探して、枠を出す
2. **SAM 2** が、その枠の中の物を切り抜く

OWLv2 は、画像と文章を同じ空間のベクトルにする CLIP をもとにした ViT で、画像のトークン1つごとに枠を1つ予測し、言葉との近さで点数を付けます。決まったクラスの一覧ではなく、自由な言葉で探せる（open-vocabulary）のが特徴です。学習には、Web の約100億枚の画像と説明文に、別の検出モデルで付けた仮の枠を使っています（M. Minderer et al., "Scaling Open-Vocabulary Object Detection," NeurIPS 2023）。

研究では、このように**得意なことが違うモデルを組み合わせて**、1つのモデルではできない処理を作ることがよくあります。初回は OWLv2 の重み（約600MB）をダウンロードします。

- `QUERY`：探す物（英語。`a cup` のように書く。カンマで区切ると複数）
- `THRESHOLD`：OWLv2 の点数のしきい値。下げると多く見つかるが、間違いも増える

In [ ]:
IMAGE = "coffee"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
QUERY = "a cup, a spoon, a plate"  # @param {type:"string"}
THRESHOLD = 0.2  # @param {type:"slider", min:0.05, max:0.6, step:0.05}

image = images[IMAGE]
queries = [q.strip() for q in QUERY.split(",") if q.strip()]
found = load_detector()(image, candidate_labels=queries, threshold=THRESHOLD)  # 1. 言葉で探して枠を出す
if not found:
    print("見つかりませんでした。THRESHOLD を下げるか、言葉を変えてみましょう")
else:
    masks, labels = [], []
    for r in found:
        b = r["box"]
        box = (b["xmin"] / (image.width - 1), b["ymin"] / (image.height - 1),
               b["xmax"] / (image.width - 1), b["ymax"] / (image.height - 1))
        masks.append(segment(image, box=box, multimask=False)[0][0])  # 2. 枠の中を切り抜く
        labels.append(f"{r['label']} ({r['score']:.2f})")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image)
    for i, r in enumerate(found):
        b = r["box"]
        color = COLORS[i % len(COLORS)] / 255
        axes[0].add_patch(plt.Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"],
                                        fill=False, color=color, lw=2.5))
        axes[0].text(b["xmin"], b["ymin"] - 5, labels[i], color="white", backgroundcolor=color, fontsize=9)
    axes[0].set_title("1. OWLv2: boxes")
    axes[1].imshow(overlay(image, masks))
    axes[1].set_title("2. SAM 2: masks")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### 5-4. 全部を自動で切り抜く

データエンジンの全自動の段階（2-3）と同じように、画像に格子状に点を並べて、1つずつ切り抜きます。品質の予測が低いものと、ほかと重なるもの（同じ物）を除くと、写っている物を全部切り抜けます。

- `GRID`：縦横に並べる点の数（16 なら 16×16 = 256 個）。増やすと小さい物まで拾えるが、時間がかかる
- `MIN_SCORE`：残す品質の予測の下限

In [ ]:
IMAGE = "motorcycle"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
GRID = 16  # @param [8, 16, 24, 32] {type:"raw"}
MIN_SCORE = 0.8  # @param {type:"slider", min:0.5, max:0.95, step:0.05}

image = images[IMAGE]
start = time.time()
masks = segment_grid(image, n=GRID, min_score=MIN_SCORE)
print(f"{GRID}×{GRID} = {GRID * GRID} 個の点 → {len(masks)} 個の物（{time.time() - start:.1f} 秒）")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].imshow(image, extent=(0, 1, 1, 0))
grid = (np.arange(GRID) + 0.5) / GRID
axes[0].plot(*np.meshgrid(grid, grid), ".", color="lime", markersize=3)
axes[0].set_title(f"{GRID}x{GRID} points")
axes[1].imshow(overlay(image, sorted(masks, key=lambda m: -m.sum())))
axes[1].set_title(f"{len(masks)} masks")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 活用：切り抜いた結果を使う

目安：約12分（うち計算 約1分）

マスクは、画素ごとに True（物）か False（物でない）の配列です。これを別の処理につなげます。

### 6-1. 背景を差し替える

切り抜いた物を、別の背景の上に置きます。指定のしかたは 5-1 と同じで、例では宇宙飛行士の人物全体を切り抜きます。マスクの境目を少しぼかすと、なじみがよくなります。`SAVE_PNG` にチェックを入れると、背景を透明にした PNG をダウンロードできます（Colab のみ）。

In [ ]:
IMAGE = "astronaut"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
POINTS = [(0.45, 0.65, 1), (0.40, 0.25, 1), (0.62, 0.86, 0)]  # (x, y, ラベル)。1：含める、0：除く
BACKGROUND = "rocket"  # @param ["rocket", "gray", "blurred"]
SAVE_PNG = False  # @param {type:"boolean"}

image = images[IMAGE]
mask = segment(image, points=POINTS, multimask=len(POINTS) == 1)[0][0]  # 品質の予測が一番高い候補
alpha = cv2.GaussianBlur(mask.astype(np.float32), (0, 0), 1.5)[..., None]  # 境目を少しぼかす
fg = np.asarray(image, dtype=np.float32)
if BACKGROUND == "rocket":  # 打ち上げ台のロケット（SpaceX、パブリックドメイン）
    bg = np.asarray(Image.fromarray(data.rocket()).resize(image.size), dtype=np.float32)
elif BACKGROUND == "gray":
    bg = np.full_like(fg, 128)
else:
    bg = cv2.GaussianBlur(fg, (0, 0), 12)
composite = (fg * alpha + bg * (1 - alpha)).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(image)
draw_prompts(axes[0], image, POINTS)
axes[0].set_title("input + points")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("mask")
axes[2].imshow(composite)
axes[2].set_title(f"new background: {BACKGROUND}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

if SAVE_PNG:
    from google.colab import files

    rgba = np.dstack([np.asarray(image), (alpha[..., 0] * 255).astype(np.uint8)])
    Image.fromarray(rgba).save(f"{IMAGE}_cutout.png")
    files.download(f"{IMAGE}_cutout.png")

### 6-2. 物だけ色を残す

切り抜いた物だけカラーのまま残し、ほかを白黒にします（カラースプラッシュ）。点を複数置くと、1つの物として扱います。

In [ ]:
IMAGE = "coffee"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
POINTS = [(0.45, 0.45, 1)]  # (x, y, ラベル) のリスト

image = images[IMAGE]
mask = segment(image, points=POINTS, multimask=len(POINTS) == 1)[0][0]
color = np.asarray(image, dtype=np.float32)
gray = np.repeat(color.mean(axis=2, keepdims=True), 3, axis=2)
splash = np.where(mask[..., None], color, gray).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(image)
draw_prompts(axes[0], image, POINTS)
axes[0].set_title("input + points")
axes[1].imshow(splash)
axes[1].set_title("color splash")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6-3. 数えて測る

5-4 の全自動の切り抜きを使えば、写っている物の**数**と、1つずつの**大きさ**（マスクの画素数）が分かります。研究では、顕微鏡の写真の細胞や粒子を数える、材料の写真で傷の面積を測る、といった使い方があります。

古代ギリシャの硬貨の写真で試します。写真には硬貨が24枚写っています。

In [ ]:
IMAGE = "coins"  # @param ["motorcycle", "astronaut", "coffee", "cat", "coins"] {allow-input: true}
GRID = 16  # @param [8, 16, 24, 32] {type:"raw"}

image = images[IMAGE]
masks = sorted(segment_grid(image, n=GRID), key=lambda m: -m.sum())
areas = [int(m.sum()) for m in masks]
print(f"見つかった物: {len(masks)} 個")
print(f"大きさ（画素数）: 最小 {min(areas)}、中央値 {int(np.median(areas))}、最大 {max(areas)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), width_ratios=[1.3, 1])
axes[0].imshow(overlay(image, masks))
for i, m in enumerate(masks):  # 大きい順に番号を付ける
    ys, xs = np.nonzero(m)
    axes[0].text(xs.mean(), ys.mean(), str(i + 1), color="white", fontsize=9, ha="center", va="center",
                 fontweight="bold")
axes[0].set_title(f"{len(masks)} objects (numbered by size)")
axes[0].axis("off")
axes[1].bar(range(1, len(areas) + 1), areas)
axes[1].set_xlabel("object number")
axes[1].set_ylabel("area [pixels]")
axes[1].set_title("area of each object")
plt.tight_layout()
plt.show()

画素数を実際の面積にするには、1画素が何 mm にあたるか（写真の縮尺）が必要です。物差しなど、大きさの分かる物を一緒に写しておくと求められます。

### 6-4. 動画で追いかける（SAM 2 の動画機能）

最初のフレームで物を1つ指定すると（ここでは枠で囲む）、SAM 2 が残りのフレームで同じ物を追いかけます。各フレームを読むたびに、それまでのフレームの結果を**メモリ**から参照するので、物の位置や見え方が変わっても追いかけられます（2-3）。

例の動画は、バイクの写真を少しずつずらしながら拡大して作った、疑似的な動画（30フレーム）です。カメラがバイクに近づきながら横に動く様子にあたります。`USE_VIDEO_UPLOAD` にチェックを入れると、自分で撮った動画（MP4 など、数秒のもの）を使えます。初回は動画用のモデルを読み込みます（重みは画像用と同じ）。

- `BOX`：最初のフレームで追いかける物を囲む枠（左上の x, 左上の y, 右下の x, 右下の y）。自分の動画では、まず `BOX` はそのままで実行し、表示される最初のフレームの目盛りを見て決め直してから、もう一度実行します

In [ ]:
USE_VIDEO_UPLOAD = False  # @param {type:"boolean"}
BOX = (0.10, 0.15, 0.92, 0.92)  # (x0, y0, x1, y1)。位置は 0〜1
MAX_FRAMES = 40  # 自分の動画は、この数のフレームに間引く

if USE_VIDEO_UPLOAD:
    from google.colab import files

    filename, content = next(iter(files.upload().items()))
    with open(filename, "wb") as f:
        f.write(content)
    capture = cv2.VideoCapture(filename)
    all_frames = []
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        all_frames.append(frame)
    picked = all_frames[::max(1, len(all_frames) // MAX_FRAMES)][:MAX_FRAMES]
    frames = [np.asarray(fit_size(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)), 640)) for f in picked]
else:  # 写真を少しずつずらしながら拡大して作る、疑似的な動画
    base = np.asarray(motorcycle)
    h, w = base.shape[:2]
    frames = []
    for k in range(30):
        s = 1 + 0.5 * k / 29  # 拡大率 1 → 1.5
        cw, ch = int(w / s), int(h / s)
        x0, y0 = int((w - cw) * (0.2 + 0.6 * k / 29)), int((h - ch) * 0.5)
        frames.append(np.asarray(Image.fromarray(base[y0:y0 + ch, x0:x0 + cw]).resize((640, 432))))
print(f"{len(frames)} フレーム、{frames[0].shape[1]}×{frames[0].shape[0]} 画素")
show_grid(Image.fromarray(frames[0]), "first frame")

video_processor = Sam2VideoProcessor.from_pretrained(SAM_ID)
video_model = Sam2VideoModel.from_pretrained(SAM_ID, dtype=dtype).to(device).eval()
session = video_processor.init_video_session(video=frames, inference_device=device, dtype=dtype)
fh, fw = frames[0].shape[:2]
video_processor.add_inputs_to_inference_session(  # 最初のフレームで、追いかける物を枠で指定する
    inference_session=session, frame_idx=0, obj_ids=1,
    input_boxes=[[[BOX[0] * (fw - 1), BOX[1] * (fh - 1), BOX[2] * (fw - 1), BOX[3] * (fh - 1)]]])

start = time.time()
tracked = {}
with torch.no_grad():
    video_model(inference_session=session, frame_idx=0)
    for output in video_model.propagate_in_video_iterator(session):  # 残りのフレームへ広げる
        mask = video_processor.post_process_masks([output.pred_masks], original_sizes=[[fh, fw]], binarize=True)[0]
        tracked[output.frame_idx] = mask[0, 0].cpu().numpy()
print(f"追いかけた時間: {time.time() - start:.1f} 秒")

shown = []
for k, frame in enumerate(frames):
    image_k = overlay(Image.fromarray(frame), [tracked[k]] if k in tracked else [])
    shown.append(np.asarray(Image.fromarray(image_k).resize((400, round(400 * fh / fw)))))
show_gif(shown)

## 7. まとめ

目安：約2分

| 分かったこと | 節 |
| --- | --- |
| セグメンテーションは画素ごとに物を分ける「密な予測」。SAM は「どれ」を指定すると、その物を切り抜く | 2-1 |
| SAM は、重い画像エンコーダ（画像ごとに1回）と、軽いプロンプトエンコーダとマスクデコーダ（指定ごと）に分かれている。だから対話的に使える | 2-2 |
| 1つの点はあいまいなので、3つの候補を品質の予測と一緒に出す。点を足す・除く、枠で囲むと、意図を伝えられる | 3.、5-1、5-2 |
| モデルとデータを一緒に育てるデータエンジンで、1100万枚・11億個のマスクを作って学習した | 2-3 |
| 言葉で探すモデル（OWLv2）と組み合わせると、言葉で切り抜ける | 5-3 |
| マスクは、背景の差し替え、色の加工、数と大きさの計測、動画での追跡などの材料になる | 6. |

**苦手なこと**（4. で見つけたものはありますか）

- 透明な物、細い物（髪の毛、針金）、境目がはっきりしない物（煙、影）
- 同じ色や模様の物がくっついているとき（どこまでが1つかが分かりにくい）
- 何の物かは答えない。「この物は何か」には、分類や検出のモデルを組み合わせる必要がある